# Vector retrieval, filtered by resolved identity

Replaces `scripts/demo_vectors.py`.

Retrieval finds text that is *relevant*; it never establishes that anything is *true*. The governance engine cannot reach the vector client — a test asserts it.

## Setup

Redacted readiness report — `config.py` never prints a secret.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")          # Neo4j emits schema hints we handle
import config
print(config.report())

readiness
  neo4j      uri=neo4j+s://<redacted>.databases.neo4j.io
             user fp=6b611cde  db fp=6b611cde
             password fp=58ebef03
             configured=True
  qdrant     url=https://<redacted>.sa-east-1-0.aws.cloud.qdrant.io
             api_key fp=f3da5a0f
             configured=True
  embedding  provider=bedrock  model=amazon.titan-embed-text-v2:0  dim=1024
             configured=True
  trust      policy=v1.0  floor=0.75  min_tier=ingested


## Fact path and retrieval path never meet

In [2]:
import ast, pathlib
pkg = pathlib.Path("../biomedical")
for sub in ("foundation", "graph"):
    bad = []
    for f in (pkg / sub).rglob("*.py"):
        tree = ast.parse(f.read_text())
        for n in ast.walk(tree):
            names = []
            if isinstance(n, ast.Import):
                names = [a.name for a in n.names]
            elif isinstance(n, ast.ImportFrom) and n.module:
                names = [n.module]
            bad += [f"{f.name} -> {x}" for x in names if "vectors" in x or "qdrant" in x.lower()]
    print(f"  {sub:<12} imports of the vector layer: {bad or 'NONE  <- correct'}")

  foundation   imports of the vector layer: NONE  <- correct
  graph        imports of the vector layer: NONE  <- correct


## Index four chunks, one of them a trap

The nivolumab chunk has near-identical wording to the pembrolizumab ones. Pure text similarity retrieves it; identity filtering does not.

In [3]:
from datetime import datetime, timezone
from biomedical.vectors.store import Chunk, VectorStore

NOW = datetime.now(timezone.utc)
PEMBRO, NIVO = "ent:drug:pembrolizumab", "ent:drug:nivolumab"
cfg = config.load()

CHUNKS = [
    Chunk("doc:label:pembro#c1", "Pembrolizumab is a PD-1 blocking antibody indicated for melanoma.",
          "dailymed", "2026-09", "label:pembro", NOW, (PEMBRO,)),
    Chunk("doc:label:pembro#c2", "Immune-mediated pneumonitis has been reported with pembrolizumab.",
          "dailymed", "2026-09", "label:pembro", NOW, (PEMBRO,)),
    Chunk("doc:label:nivo#c1", "Nivolumab is a PD-1 blocking antibody indicated for melanoma.",
          "dailymed", "2026-09", "label:nivo", NOW, (NIVO,)),       # the trap
    Chunk("doc:trial:NCT1#c1", "Open-label phase 3 trial of checkpoint inhibition in melanoma.",
          "clinicaltrials", "2026-09", "NCT1", NOW, ()),            # unlinked
]

store = VectorStore(url=cfg.qdrant.url, api_key=cfg.qdrant.api_key.reveal(),
                    embedding_model=cfg.embedding.model_id, dim=cfg.embedding.dim)
with store as s:
    print("collection:", s.collection)
    if not cfg.embedding.configured:
        print("embedding provider NONE -> deterministic stand-in.")
        print("Structural behaviour below is real; semantic ranking is NOT meaningful")
        print("and no retrieval-quality claim may be made from it.")
    s.ensure_collection(recreate=True)
    print("\nindexed:", s.index(CHUNKS), " re-indexed:", s.index(CHUNKS),
          " points:", s.client.count(s.collection).count, "(idempotent)")

collection: biomedv2__amazon-titan-embed-text-v2-0__chunk-v1



indexed: 4  re-indexed: 4  points: 4 (idempotent)


## Unfiltered vs identity-filtered

`identity_grounded=False` is a first-class signal, not an absence. A caller must be able to tell *"filtered to this entity"* from *"we could not resolve it, so this is a text match"*.

In [4]:
query = "PD-1 blocking antibody for melanoma"
with store as s:
    unfiltered = s.retrieve(query, limit=4)
    filtered = s.retrieve(query, entity_ids=(PEMBRO,), limit=4)

print("1) unfiltered —", unfiltered.describe())
print("   note:", unfiltered.note)
for c in unfiltered.chunks:
    print(f"     {c.score:+.4f}  {c.chunk_id:<24} {list(c.entity_ids) or '[]'}")

print("\n2) filtered —", filtered.describe())
for c in filtered.chunks:
    print(f"     {c.score:+.4f}  {c.chunk_id:<24} {list(c.entity_ids)}")

returned = {c.chunk_id for c in filtered.chunks}
print()
print("   nivolumab trap excluded ....", "doc:label:nivo#c1" not in returned)
print("   unlinked chunk excluded ....", "doc:trial:NCT1#c1" not in returned)
print("   identity_grounded ..........", filtered.identity_grounded)

1) unfiltered — 4 chunk(s) · dense · UNFILTERED (identity unresolved)
   note: no entity resolved; results are text-similarity only
     +0.0457  doc:label:pembro#c1      ['ent:drug:pembrolizumab']
     +0.0247  doc:label:nivo#c1        ['ent:drug:nivolumab']
     +0.0127  doc:trial:NCT1#c1        []
     -0.0088  doc:label:pembro#c2      ['ent:drug:pembrolizumab']

2) filtered — 2 chunk(s) · dense · identity-grounded · filters=['entity_ids∈ent:drug:pembrolizumab']
     +0.0457  doc:label:pembro#c1      ['ent:drug:pembrolizumab']
     -0.0088  doc:label:pembro#c2      ['ent:drug:pembrolizumab']

   nivolumab trap excluded .... True
   unlinked chunk excluded .... True
   identity_grounded .......... True


## Degradation is visible, not silent

An unreachable store must never return an empty list that reads as "nothing matched".

In [5]:
broken = VectorStore(url="https://127.0.0.1:1", api_key="none", dim=cfg.embedding.dim, timeout=2)
with broken as b:
    r = b.retrieve(query, entity_ids=(PEMBRO,))
print(r.describe())
print("note:", r.note)
print("degraded flag set rather than a bare empty result:", r.degraded)

with store as s:
    s.drop()
print("\ncleanup: dropped", store.collection)

0 chunk(s) · dense · DEGRADED · filters=['entity_ids∈ent:drug:pembrolizumab']
note: vector store unavailable: ResponseHandlingException
degraded flag set rather than a bare empty result: True



cleanup: dropped biomedv2__amazon-titan-embed-text-v2-0__chunk-v1
